In [ ]:
from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from samos.shmodel import SHModel
%matplotlib inline

FILE = Path('/project/samos/2023_gji_gnm_GOVO/2023_gji_gnm_GOVO.hdf5')
a, R_C = 6371.2, 3485.0          # Earth reference radius and CMB radius [km], as in pygeodyn
PC = ccrs.PlateCarree()

In [ ]:
def print_tree(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f'  {name:<20} {str(obj.shape):<18} {obj.dtype}')
    else:
        print(f'{name}/')

with h5py.File(FILE, 'r') as f:
    f.visititems(print_tree)
    attrs = dict(f.attrs)
    data = {k: f['analysed'][k][()] for k in f['analysed']}      # ~190 MB, all in memory

print(f'\n{len(attrs)} root attributes')

The dataset axes are always **(realisation, time, coefficient)**:
- **400 realisations**: the ensemble members. Their spread is the uncertainty of the reanalysis; the ensemble mean is the best estimate.
- **78 times**: the *analysis* times, every 4 months (`dt_a`) from 1997.08 to 2022.75. Between analyses the filter runs monthly
  forecasts (attribute `t_forecasts`), but those are **not stored**; only the `analysed` group is in the file.
- **coefficients**: spherical harmonic coefficients (section 3).

## 2. Run configuration (root attributes)

In [ ]:
def decode(K):
    """pygeodyn ordering -> degree n, order m, is_sine for K coefficients."""
    n, m, s = [], [], []
    l = 0
    while len(n) < K:
        l += 1
        n.append(l); m.append(0); s.append(False)
        for mm in range(1, l + 1):
            n += [l, l]; m += [mm, mm]; s += [False, True]
    return np.array(n[:K]), np.array(m[:K]), np.array(s[:K])

def flow_matrices(L, colat, lon):
    """Matrices A_th, A_ph (Npoints, 2 L(L+2)) so that u_theta = A_th @ U, u_phi = A_ph @ U [km/yr].
    colat, lon in degrees, 1D arrays of the same length (points)."""
    K = L * (L + 2)
    n, m, s = decode(K)
    idx = n * (n + 1) // 2 + m
    nt = (L + 1) * (L + 2) // 2
    P, dP, Ps = SHModel(*[np.zeros((1, nt))] * 4, [2000.0], L, 2001.0)._legendre(colat)
    P, dP, Ps = P[:, idx], dP[:, idx], Ps[:, idx]                  # (Npoints, K)
    mphi = np.radians(lon)[:, None] * m
    trig = np.where(s, np.sin(mphi), np.cos(mphi))                 # Y = P * trig
    dtrig = np.where(s, np.cos(mphi), -np.sin(mphi)) * m           # d(trig)/dphi
    dY_dth = dP * trig
    dY_dph_sin = Ps * dtrig                                        # (1/sin theta) dY/dphi
    A_th = np.hstack((dY_dph_sin, dY_dth))                         # [toroidal | poloidal]
    A_ph = np.hstack((-dY_dth, dY_dph_sin))
    return A_th, A_ph

times = data['times'].astype(float)
months = np.round((times - 1970) * 12).astype(int)           # months since 1970-01
dates = np.datetime64('1970-01', 'M') + months              # exact month starts
epochs = 1970 + months / 12                                  # exact decimal years
print(dates[:4], '...', dates[-2:])
print('spacing [months]:', np.unique(np.diff(months)))
it = len(dates) - 1                                                   # last analysis
print('time:', dates[it]) 
lat = np.arange(-89, 90, 2.0); lon = np.arange(-179, 180, 2.0)       # 2-degree grid (cell centres)
LAT, LON = np.meshgrid(lat, lon, indexing = 'ij')
U = data['U']                                                        # (400, 78, 720)
A_th, A_ph = flow_matrices(18, 90 - LAT.ravel(), LON.ravel())
u_th_ens = (U[:, it] @ A_th.T).reshape(400, *LAT.shape)             # all realisations at once
u_ph_ens = (U[:, it] @ A_ph.T).reshape(400, *LAT.shape)
u_th, u_ph = u_th_ens.mean(0), u_ph_ens.mean(0)

# w = np.cos(np.radians(LAT))                                          # area weights
# rms = lambda x: np.sqrt((x * w).sum(axis = (-2, -1)) / w.sum())
# print(f'rms speed, ensemble-mean flow      : {rms(u_th**2 + u_ph**2):.1f} km/yr')
# print(f'rms speed, individual realisations : {rms(u_th_ens**2 + u_ph_ens**2).mean():.1f} km/yr')

# fig, axs = plt.subplots(1, 2, figsize = (12, 3.5), layout = 'constrained')
# axs[0].plot(lat, u_ph.mean(1), 'k')
# axs[0].axhline(0, color = 'gray', lw = 0.5)
# axs[0].set(title = r'zonal mean of $u_\phi$ (negative = westward)', xlabel = 'latitude', ylabel = 'km/yr')
# Tz = U[:, :, 0]
# axs[1].plot(epochs, Tz.mean(0), 'k')
# axs[1].fill_between(epochs, *np.percentile(Tz, [5, 95], axis = 0), alpha = 0.3)
# axs[1].set(title = r'$t_1^0$ (solid-body rotation of the core surface), 5-95%', xlabel = 'year', ylabel = 'km/yr')
# plt.show()

The ensemble-mean flow is smoother and weaker than any single realisation: averaging cancels the small, uncertain scales.
For maps it is usually the ensemble mean that is shown; the realisations show what is (not) constrained.

In [ ]:
speed = np.hypot(u_th, u_ph)
speed_std = np.hypot(u_th_ens, u_ph_ens).std(0)

def map_axes(n, title_list, figsize = (14, 4)):
    fig, axs = plt.subplots(1, n, figsize = figsize, layout = 'constrained',
                            subplot_kw = {'projection': ccrs.Mollweide()})
    for ax, t in zip(np.atleast_1d(axs), title_list):
        ax.set_global(); ax.coastlines(lw = 0.5, color = 'gray'); ax.set_title(t)
    return fig, np.atleast_1d(axs)



fig, axs = map_axes(2, [f'flow speed, ensemble mean, {dates[it]}', 'ensemble std of the speed'])
m0 = axs[0].pcolormesh(lon, lat, speed, transform = PC, cmap = 'magma_r')
# regrid_shape: cartopy re-samples the arrows evenly in the projection (no crowding at the poles)
axs[0].quiver(LON, LAT, u_ph, -u_th, transform = PC, regrid_shape = 25, scale = 500, width = 0.002)
m1 = axs[1].pcolormesh(lon, lat, speed_std, transform = PC, cmap = 'viridis')
fig.colorbar(m0, ax = axs[0], orientation = 'horizontal', shrink = 0.6, label = 'km/yr')
fig.colorbar(m1, ax = axs[1], orientation = 'horizontal', shrink = 0.6, label = 'km/yr')
plt.show()